# Qwen3-VL-30B-A3B LoRA Fine-tuning (H100 최적화)

- **모델**: `Qwen/Qwen3-VL-30B-A3B-Instruct` (MoE 구조 · 30B 파라미터 · 3B 활성화)
- **방법**: LoRA (rank=32) — 80GB VRAM 내에서 안정적으로 학습
- **환경**: H100 80GB (Colab Pro+)
- **베이스**: Qwen3-VL-8B Full FT 코드에서 변환

# 0. GPU 확인

In [ ]:
# GPU 확인 — H100이 배정됐는지 먼저 확인!
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU 정보:', result.stdout.strip())

gpu_name = result.stdout.strip()
if 'H100' in gpu_name:
    print('✅ H100 확인! 30B LoRA 설정으로 진행합니다.')
elif 'A100' in gpu_name:
    print('⚠️  A100 감지. BATCH_SIZE=2, GRAD_ACCUM=8 로 변경을 권장합니다.')
else:
    print('❌ 주의: H100이 아닙니다. VRAM 부족으로 OOM 가능성이 높습니다.')

# 1. 환경 설치

In [ ]:
# ✅ [30B 변경] peft 추가 — LoRA에 필요
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q "accelerate>=0.34.2" "peft>=0.13.2" bitsandbytes datasets pillow pandas qwen-vl-utils --upgrade

In [ ]:
# Flash Attention 2 설치 (H100 속도 향상)
!pip install -q ninja packaging psutil
!pip install -v flash-attn

In [ ]:
import torch
print('Torch 버전:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

# 2. 라이브러리 · 설정

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from collections import Counter

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)
# ✅ [30B 변경] LoRA 관련 import 추가
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import bitsandbytes as bnb
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# ============================
# 실험 설정
# ============================
EXP_NUM    = 1           # ← 새 실험마다 여기만 변경!

# ✅ [30B 변경] 모델 ID 변경
MODEL_ID   = 'Qwen/Qwen3-VL-30B-A3B-Instruct'

IMAGE_SIZE = 512
MAX_NEW_TOKENS = 8
SEED       = 42

# ✅ [30B 변경] 하이퍼파라미터 조정
# 30B 모델은 메모리 사용량이 크므로 배치 줄이고 누적 늘림
BATCH_SIZE   = 4          # H100 80GB 기준 (8B Full FT의 16 → 4로 감소)
GRAD_ACCUM   = 4          # 유효 배치 = 4 × 4 = 16 (8B와 동일한 유효 배치 유지)
NUM_EPOCHS   = 3
LR           = 2e-4       # LoRA는 Full FT보다 LR 높게 설정
WARMUP_RATIO = 0.05

# ✅ [30B 변경] LoRA 하이퍼파라미터
LORA_R       = 32         # rank — 높을수록 성능↑ VRAM↑
LORA_ALPHA   = 64         # alpha = 2 × r 권장
LORA_DROPOUT = 0.05

BASE_DIR = '/content'
SAVE_DIR = f'{BASE_DIR}/qwen3_30b_lora_exp{EXP_NUM}_best'
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# H100 전용 TF32 활성화
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

train_df = pd.read_csv(f'{BASE_DIR}/train.csv')
test_df  = pd.read_csv(f'{BASE_DIR}/test.csv')

# answer 결측치 제거 (EDA에서 1개 확인됨)
train_df = train_df.dropna(subset=['answer']).reset_index(drop=True)

print(f'학습 데이터: {len(train_df)}개')
print(f'테스트 데이터: {len(test_df)}개')
print(f'저장 경로: {SAVE_DIR}')
print(f'유효 배치 크기: {BATCH_SIZE * GRAD_ACCUM}')

# 3. 모델 · Processor 로드

> ⏳ 30B 모델 다운로드: 약 60GB · 15~30분 소요

In [ ]:
import flash_attn

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

# ✅ [30B 변경] bfloat16으로 로드 (80GB VRAM에 ~62GB 사용)
# MoE 구조라 전체 30B 가중치가 메모리에 올라오지만 연산은 3B만 활성화됨
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2',
    device_map='auto',           # 80GB 내에서 자동 배치
    trust_remote_code=True,
)

# ✅ [30B 변경] LoRA를 위한 준비
# kbit 양자화 없이 사용하므로 prepare_model_for_kbit_training 대신
# gradient_checkpointing만 활성화
base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
base_model.enable_input_require_grads()  # LoRA + gradient checkpointing 호환

# ✅ [30B 변경] LoRA 설정
# MoE 모델의 경우 MoE expert 레이어(gate_proj, up_proj, down_proj)도 포함
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    task_type='CAUSAL_LM',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n전체 파라미터: {total_params/1e9:.2f}B')
print(f'학습 가능 파라미터: {trainable_params/1e6:.1f}M ({100*trainable_params/total_params:.2f}%)')
print(f'저장 경로: {SAVE_DIR}')

## (선택) 이전 체크포인트 이어 학습

커널 재시작 후 이어서 학습할 때만 실행

In [ ]:
# from peft import PeftModel
#
# LOAD_DIR = f'{BASE_DIR}/qwen3_30b_lora_exp{EXP_NUM}_best'
# model = PeftModel.from_pretrained(base_model, LOAD_DIR)
# model.train()
# print(f'체크포인트 로드 완료: {LOAD_DIR}')

# 4. 프롬프트 템플릿

In [ ]:
SYSTEM_INSTRUCT = (
    "당신은 재활용품 분류를 위한 시각적 질의응답 어시스턴트입니다. "
    "답변하기 전에 반드시 이미지를 주의 깊게 살펴보아야 합니다. "
    "텍스트의 길이나 단어 패턴을 보고 추측하지 마세요. "
    "이미지에 실제로 보이는 물체, 재질, 라벨에만 집중하십시오. "
    "설명 없이 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 답변하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f'{question}\n'
        f'(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n'
        '정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.'
    )

# 5. Dataset · DataCollator

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['path']).convert('RGB')
        q = str(row['question'])
        a, b, c, d = str(row['a']), str(row['b']), str(row['c']), str(row['d'])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user',   'content': [
                {'type': 'image', 'image': img},
                {'type': 'text',  'text': user_text}
            ]}
        ]
        if self.train:
            gold = str(row['answer']).strip().lower()
            messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': gold}]})

        return {'messages': messages, 'image': img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample['messages'], tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(sample['image'])

        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors='pt'
        )

        if self.train:
            labels = enc['input_ids'].clone()
            assistant_token_ids = self.processor.tokenizer.encode(
                '<|im_start|>assistant\n', add_special_tokens=False
            )
            for i, label_row in enumerate(labels):
                mask_until = len(label_row)
                for j in range(len(label_row) - len(assistant_token_ids)):
                    if label_row[j:j+len(assistant_token_ids)].tolist() == assistant_token_ids:
                        mask_until = j + len(assistant_token_ids)
                        break
                labels[i, :mask_until] = -100
            labels[enc['attention_mask'] == 0] = -100
            enc['labels'] = labels

        return enc

# 6. DataLoader

In [ ]:
train_df_shuffled = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split = int(len(train_df_shuffled) * 0.9)
train_subset = train_df_shuffled.iloc[:split].reset_index(drop=True)
valid_subset = train_df_shuffled.iloc[split:].reset_index(drop=True)

print(f'Train: {len(train_subset)}개, Valid: {len(valid_subset)}개')

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# ✅ [30B 변경] 배치 크기 4 (8B Full FT의 16에서 감소)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=4, pin_memory=True
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=4, pin_memory=True
)
print(f'Train 배치 수: {len(train_loader)}, Valid 배치 수: {len(valid_loader)}')

# 7. Fine-tuning

> ⏳ 예상 학습 시간: epoch당 약 40~60분 (H100 기준)

In [ ]:
from tqdm.auto import tqdm

def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return 'a'
    last = lines[-1]
    if last in ['a', 'b', 'c', 'd']:
        return last
    for line in reversed(lines):
        for tok in line.split():
            tok_clean = tok.strip('.,!?()[]')
            if tok_clean in ['a', 'b', 'c', 'd']:
                return tok_clean
    matches = re.findall(r'\b([abcd])\b', text)
    if matches:
        return matches[-1]
    counts = Counter(re.findall(r'[abcd]', text))
    if counts:
        return counts.most_common(1)[0][0]
    return 'a'


model = model.to(device)
AMP_DTYPE = torch.bfloat16

# ✅ [30B 변경] 8-bit AdamW 유지 — LoRA 파라미터 수는 적지만 optimizer 상태 절약
optimizer = bnb.optim.AdamW8bit(
    filter(lambda p: p.requires_grad, model.parameters()),  # LoRA 파라미터만
    lr=LR,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

num_training_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * WARMUP_RATIO),
    num_training_steps=num_training_steps
)

best_val_acc = 0.0
global_step  = 0

print(f'총 학습 스텝: {num_training_steps}')
print(f'Warmup 스텝: {int(num_training_steps * WARMUP_RATIO)}')
print(f'배치 크기: {BATCH_SIZE}, Grad Accum: {GRAD_ACCUM}, 유효 배치: {BATCH_SIZE * GRAD_ACCUM}')
print(f'LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}')
print('─' * 50)

for epoch in range(NUM_EPOCHS):
    # ───────────────── Train ─────────────────
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]', unit='batch')

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            # ✅ LoRA 파라미터만 clipping
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            current_lr = scheduler.get_last_lr()[0]
            progress_bar.set_postfix({
                'loss': f'{running_loss:.3f}',
                'lr': f'{current_lr:.2e}'
            })
            running_loss = 0.0

    # ───────────────── Validation ─────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for row in tqdm(valid_subset.itertuples(), total=len(valid_subset),
                        desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [valid]'):
            img = Image.open(row.path).convert('RGB')
            user_text = build_mc_prompt(row.question, row.a, row.b, row.c, row.d)
            messages = [
                {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
                {'role': 'user',   'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': user_text}]}
            ]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[text], images=[img], return_tensors='pt').to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                out_ids = model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    eos_token_id=processor.tokenizer.eos_token_id
                )
            pred = extract_choice(processor.batch_decode(out_ids, skip_special_tokens=True)[0])
            correct += int(pred == str(row.answer).strip().lower())

            del inputs
            torch.cuda.empty_cache()

    val_acc = correct / len(valid_subset)
    print(f'[Epoch {epoch+1}] Valid Acc: {val_acc:.4f} ({correct}/{len(valid_subset)})')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # ✅ [30B LoRA] LoRA 어댑터만 저장 (전체 모델 대신 수백 MB만 저장)
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f'  ★ Best model saved (acc={best_val_acc:.4f}) → {SAVE_DIR}')

    model.train()

print(f'\n학습 완료. Best Valid Acc: {best_val_acc:.4f}')

# 8. Inference

In [ ]:
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc='Inference', unit='sample'):
    row = test_df.iloc[i]
    try:
        img = Image.open(row['path']).convert('RGB')
        user_text = build_mc_prompt(row['question'], row['a'], row['b'], row['c'], row['d'])

        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
            {'role': 'user',   'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': user_text}]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors='pt').to(device)

        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out_ids = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id
            )
        output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
        preds.append(extract_choice(output_text))

    except Exception as e:
        print(f'[Sample {i}] 에러: {e}')
        preds.append('a')

    finally:
        del inputs
        torch.cuda.empty_cache()

# 제출 파일 생성
submission = pd.DataFrame({'id': test_df['id'], 'answer': preds})
submission.to_csv(f'{BASE_DIR}/submission.csv', index=False)

print('Saved submission.csv')
print(f'경로: {BASE_DIR}/submission.csv')
print(f'Answer distribution:\n{submission["answer"].value_counts()}')

## (선택) 응답 예시 확인

In [ ]:
print(output_text)